<a href="https://colab.research.google.com/github/Diego-Cano-bit/CIMAT-Monterrey/blob/main/CNN_ECG(86%25).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

#el framework tensorflow nos ayuda a construir de forma rápida y eficiente una red neuronal
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Dropout, Flatten

from tensorflow.keras.utils import to_categorical
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter)
from dataclasses import dataclass


In [ ]:
import scipy
from scipy.signal import butter,lfilter,freqz
from scipy.signal import find_peaks
def butter_lowpass(cutoff, fs, order=5):
    return scipy.signal.butter(order, cutoff, fs=fs, btype='low', analog=False)

def butter_lowpass_filter(data, cutoff, fs, order=5):
    b, a = butter_lowpass(cutoff, fs, order=order)
    y = scipy.signal.lfilter(b, a, data)
    return y

In [ ]:
import pathlib #librería para cargar datos
import numpy as np
import pandas as pd
import re
ejemplo_dir = '/content/drive/MyDrive/Colab Notebooks/PYTHON/MUESTRAS' #carpeta donde están los datos
#tiene que estar en la nube con permisos ilimitados
directorio = pathlib.Path(ejemplo_dir)
nombres=[]
for fichero in directorio.iterdir():
    nombres.append(fichero.name)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#se ordenan los nombres
r = re.compile(r"(\d+)")
nombres.sort(key=lambda x: int(r.search(x).group(1)))
print(nombres)

['0001.txt', '0002.txt', '0003.txt', '0004.txt', '0005.txt', '0006.txt', '0008.txt', '0009.txt', '0010.txt', '0011.txt', '0012.txt', '0013.txt', '0014.txt', '0015.txt', '0016.txt', '0017.txt', '0018.txt', '0019.txt', '0020.txt', '0021.txt', '0022.txt', '0023.txt', '0024.txt', '0025.txt', '0026.txt', '0027.txt', '0028.txt', '0029.txt', '0030.txt', '0031.txt', '0032.txt', '0033.txt', '0034.txt', '0035.txt', '0036.txt', '0037.txt', '0038.txt', '0039.txt', '0040.txt', '0041.txt', '0042.txt', '0043.txt', '0044.txt', '0045.txt', '0046.txt', '0047.txt', '0048.txt', '0049.txt', '0050.txt', '0051.txt', '0052.txt', '0053.txt', '0054.txt', '0055.txt', '0056.txt', '0057.txt', '0058.txt', '0059.txt', '0060.txt', '0061.txt', '0062.txt', '0063.txt', '0064.txt', '0065.txt', '0066.txt', '0067.txt', '0068.txt', '0069.txt', '0070.txt', '0071.txt', '0072.txt', '0073.txt', '0074.txt', '0075.txt', '0076.txt', '0077.txt', '0078.txt', '0079.txt', '0080.txt', '0081.txt', '0082.txt', '0083.txt', '0084.txt', '00

In [ ]:
señales=[] #nuevo arreglo de señales ordenados

for i in range(0,len(nombres)):
 for fichero in directorio.iterdir():
   if fichero.name==nombres[i]:
   # print("true")
    datos=pd.read_csv(fichero,names=[fichero.name])
    señales.append(datos)

In [ ]:
def image_to_array(imagen):
    size = (128, 128)
    fig = plt.figure(figsize=(size[1]/100, size[0]/100))
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)

    plt.specgram(np.array(imagen).flatten(), Fs=195, cmap="gray")

    fig.canvas.draw()
    arr = np.array(fig.canvas.renderer.buffer_rgba())
    gray_arr = np.dot(arr[..., :3], [0.2989, 0.5870, 0.1140])

    # Redimensiona el arreglo al tamaño deseado
    resized_arr = np.resize(gray_arr, size)
    plt.close(fig)
    # Retorna el arreglo redimensionado
    return resized_arr

In [ ]:
final=[]
for i in range(0,len(señales)):
  cutoff = 3.667 #cutoff frequency in rad/s
  fs = 14 #sampling frequency in rad/s
  order = 20 #order of filter
  sr = 195 #sample rate
  filtro=butter_lowpass_filter(señales[i], cutoff, fs, order)
  smooth_muestra = scipy.signal.savgol_filter(filtro, 21, 7, mode='nearest')
  final.append(image_to_array(smooth_muestra))

<ipython-input-43-11645f5bca8f>:8: UserWarning: Only one segment is calculated since parameter NFFT (=256) >= signal length (=81).
  plt.specgram(np.array(imagen).flatten(), Fs=195, cmap="gray")
<ipython-input-43-11645f5bca8f>:8: UserWarning: Only one segment is calculated since parameter NFFT (=256) >= signal length (=67).
  plt.specgram(np.array(imagen).flatten(), Fs=195, cmap="gray")
/usr/local/lib/python3.10/dist-packages/matplotlib/axes/_axes.py:7773: RuntimeWarning: divide by zero encountered in log10
  Z = 10. * np.log10(spec)


In [ ]:
final=np.array(final)/255

In [ ]:
final.shape

(786, 128, 128)

In [ ]:
y=pd.read_excel("/content/DATOS SELECCIONADOS.xlsx",header=None).dropna()
y=np.array(y).flatten()


In [ ]:
from sklearn.model_selection import train_test_split

# Supongamos que tienes tus datos almacenados en las listas o arrays X e y (características y etiquetas, respectivamente)

# Primero, dividimos los datos en train y el resto (combinado de test y validation)
X_train, X_temp, y_train, y_temp = train_test_split(final, y, test_size=0.3, random_state=42)

# Luego, dividimos el conjunto restante (test y validation) en partes iguales para test y validation
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [ ]:
def plot_results(metrics, title=None, ylabel=None, ylim=None, metric_name=None, color=None):

    fig, ax = plt.subplots(figsize=(15, 4))

    if not (isinstance(metric_name, list) or isinstance(metric_name, tuple)):
        metrics = [metrics,]
        metric_name = [metric_name,]

    for idx, metric in enumerate(metrics):
        ax.plot(metric, color=color[idx])
    #se hace la grafica
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xlim([0, TrainingConfig.EPOCHS-1])
    plt.ylim(ylim)

    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%d'))
    ax.xaxis.set_minor_locator(MultipleLocator(1))
    plt.grid(True)
    plt.legend(metric_name)
    plt.show()
    plt.close()

In [ ]:
import tensorflow.keras as keras
import tensorflow.keras.layers as layers

model = keras.Sequential([
    # Block One
    layers.Conv2D(filters=150, kernel_size=3, activation='relu', padding='same',
                  input_shape=[128, 128, 1]),
    layers.MaxPool2D(),

    # Block Two
    layers.Conv2D(filters=250,kernel_size=3, activation='relu', padding='same'),
    layers.MaxPool2D(),
    layers.Dropout(0.2),

    # Block Two
    layers.Conv2D(filters=120,kernel_size=3, activation='relu', padding='same'),
    layers.MaxPool2D(),
    layers.Dropout(0.2),

    # Block Thre
    layers.Conv2D(filters=150, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPool2D(),
    layers.Dropout(0.2),

    # Head
    layers.Flatten(),
    layers.Dense(1024, activation='sigmoid'),
    layers.Dropout(0.5),
    layers.Dense(1024, activation='sigmoid'),
    layers.Dropout(0.5),


    layers.Dense(1, activation='sigmoid'),
])
model.compile(
    optimizer="RMSprop",
    loss='binary_crossentropy',
    metrics=['binary_accuracy'],
)


In [ ]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_4 (Conv2D)           (None, 128, 128, 150)     1500      
                                                                 
 max_pooling2d_4 (MaxPooling  (None, 64, 64, 150)      0         
 2D)                                                             
                                                                 
 conv2d_5 (Conv2D)           (None, 64, 64, 250)       337750    
                                                                 
 max_pooling2d_5 (MaxPooling  (None, 32, 32, 250)      0         
 2D)                                                             
                                                                 
 dropout_5 (Dropout)         (None, 32, 32, 250)       0         
                                                                 
 conv2d_6 (Conv2D)           (None, 32, 32, 120)      

In [ ]:
history = model.fit(X_train,
                            y_train,
                            epochs=182,
                            verbose=1,
                            validation_data=(X_val, y_val),
                           )

Epoch 1/182
18/18 [==============================] - 9s 251ms/step - loss: 1.2280 - binary_accuracy: 0.5127 - val_loss: 1.5623 - val_binary_accuracy: 0.6186
Epoch 2/182
18/18 [==============================] - 2s 110ms/step - loss: 0.9379 - binary_accuracy: 0.5364 - val_loss: 0.9722 - val_binary_accuracy: 0.3814
Epoch 3/182
18/18 [==============================] - 2s 109ms/step - loss: 0.8081 - binary_accuracy: 0.5000 - val_loss: 0.6809 - val_binary_accuracy: 0.6186
Epoch 4/182
18/18 [==============================] - 2s 108ms/step - loss: 0.8156 - binary_accuracy: 0.5345 - val_loss: 0.8368 - val_binary_accuracy: 0.3814
Epoch 5/182
18/18 [==============================] - 2s 103ms/step - loss: 0.7653 - binary_accuracy: 0.5527 - val_loss: 0.6649 - val_binary_accuracy: 0.6186
Epoch 6/182
18/18 [==============================] - 2s 103ms/step - loss: 0.7562 - binary_accuracy: 0.5709 - val_loss: 0.8095 - val_binary_accuracy: 0.6186
Epoch 7/182
18/18 [==============================] - 2s 10

In [ ]:

#Samuel Romero
# Se obtienen y grafican los valores de la función de costo y el accuracy
train_loss = history.history["loss"]
train_acc  = history.history["binary_accuracy"]
valid_loss = history.history["val_loss"]
valid_acc  = history.history["val_binary_accuracy"]

#plot_results([ train_loss, valid_loss ], ylabel="Loss", ylim = [0.0, 5.0], metric_name=["Training Loss", "Validation Loss"], color=["g", "b"]);

#plot_results([ train_acc, valid_acc ], ylabel="Accuracy", ylim = [0.0, 1.0], metric_name=["Training Accuracy", "Validation Accuracy"], color=["g", "b"])

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc*100:.3f}")

4/4 [==============================] - 0s 26ms/step - loss: 0.4055 - binary_accuracy: 0.8644
Test accuracy: 86.441


In [ ]:
predictions = model.predict(X_test)
predicted_labels = [np.argmax(i) for i in predictions]

4/4 [==============================] - 0s 23ms/step


In [ ]:
from sklearn import metrics
confusion_matrix = metrics.confusion_matrix(predictions(), predicted_labels)
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix)
cm_display.plot()
cm_display.ax_.set_title("Matriz de confusión")
cm_display.ax_.set_xlabel("Predichos")
cm_display.ax_.set_ylabel("Reales")
plt.show()


TypeError: ignored